In [51]:
import re
import numpy as np

from collections import defaultdict
from editdistance import eval as distance

def del_parentheses(text):
    pattern = r"\([^()]*\)"
    return re.sub(pattern, "", text)

def del_space(text):
    pattern = r"\s+"
    return re.sub(pattern, " ", text).strip()

def del_numbering(text):
    pattern = r"^(?:\d+[\.\)、]?\s*[\-\—\–]?\s*)?"
    return re.sub(pattern, "", text)

def is_in(text, items, threshold):
    for i in items:
        if (distance(i.lower(), text.lower()) <= threshold):
            return True
    return False

def nearest(text, items):
    """ given the raw text name and all candidates, 
        return {movie_name:, min_edit_distance: , nearest_movie: }
    """
    # calculate the edit distance
    items = list(set(items))
    dists = [distance(text.lower(), i.lower()) for i in items]
    # find the nearest movie
    nearest_idx = np.argmin(dists)
    nearest_movie = items[nearest_idx]
    return {
        'movie_name': text, 
        'min_edit_distance': dists[nearest_idx], 
        'nearest_movie': nearest_movie
    }


def extract_movie_name(text):
    text = text.split('/')[-1]
    text = text.replace('_', ' ').replace('-', ' ').replace('>', ' ')
    return del_space(del_parentheses(text))

def recall_score(gt_list, pred_list, ks, threshold, verbose=False):
    hits = defaultdict(list)
    for gt, preds in zip(gt_list, pred_list):
        for k in ks:
            hits[k].append(int(is_in(gt, preds[:k], threshold)))
    if verbose:
        for k in ks:
            print("Recall@{}: {:.4f}".format(k, np.mean(hits[k])))
    return hits
    

def mrr_score(gt_list, pred_list, ks, threshold, verbose=False):
    mrrs = defaultdict(list)
    for gt, preds in zip(gt_list, pred_list):
        for k in ks:
            for i, p in enumerate(preds[:k]):
                if is_in(gt, [p], threshold):
                    mrrs[k].append(1 / (i + 1))
                else:
                    mrrs[k].append(0)
    if verbose:
        for k in ks:
            print("MRR@{}: {:.4f}".format(k, np.mean(mrrs[k])))
    return mrrs

def ndcg_score(gt_list, pred_list, ks, threshold, verbose=False):
    ndcgs = defaultdict(list)
    for gt, preds in zip(gt_list, pred_list):
        for k in ks:
            for i, p in enumerate(preds[:k]):
                if is_in(gt, [p], threshold):
                    ndcgs[k].append(1 / np.log2(i + 2))
                    break
    if verbose:
        for k in ks:
            print("NDCG@{}: {:.4f}".format(k, np.mean(ndcgs[k])))
    return ndcgs

In [52]:
import os
import sys 
import json
from jsonargparse import CLI
from tqdm import tqdm

sys.path.append('./')

DIR = os.getcwd()

def extract_list(l, candidates=None):
    text = l
    l = {}
    try:
        preference, text = text.split('1.', maxsplit=1)
    except Exception as e:
        print(e)
        preference = ""
        text = text.replace(',', '\n')
    rec_list = [del_numbering(del_space(del_parentheses(i.strip()))) for i in text.split('\n')]
    if candidates is not None:
        rec_list = [nearest(i, candidates) for i in rec_list]
    l['rec_list'] = rec_list
    l['preference'] = preference
    return l

def condition(keyword, rec, prev_entity):
    if keyword == 'recommendation':
        return rec not in prev_entity
    elif keyword == 'discussion':
        return rec in prev_entity
    else:
        return True

In [53]:
dataset = 'inspired'
model = "llama3-2-1b-instruct"
# get paths
alg = 'sft_generated'
file_name = '_test.jsonl' if alg == 'vanilla' else '_test_conv_entropy_ep5lr2e-5.jsonl' 
pred_json = f'test_res/{alg}/{dataset}/{model}/{dataset}{file_name}'
gt_json = os.path.join(DIR, f'testsets/{dataset}/test.jsonl')
meta_json = os.path.join(DIR, f'testsets/{dataset}/entity2id.json')

print(pred_json)

# load pred_json
preds = [json.loads(l) for l in open(pred_json)]

# load gt_json
gts = [json.loads(l) for l in open(gt_json)][:len(preds)]
gts = {i: g for i, g in enumerate(gts)}

# load meta_json
name2id = json.load(open(meta_json))
id2name = {v: extract_movie_name(k) for k, v in name2id.items()}

# get candidates
candidates = list(id2name.values())
# pred_list = [extract_list(l, candidates) for l in tqdm(preds)]
from multiprocessing import Pool
from functools import partial

def parallel_extract(preds, candidates=None, num_processes=None):
    extract_with_candidates = partial(extract_list, candidates=candidates)
    with Pool(processes=num_processes) as pool:
        pred_list = list(tqdm(
            pool.imap(extract_with_candidates, preds),
            total=len(preds),
            desc="Processing"
        ))
    return pred_list

pred_list = parallel_extract(preds, candidates, num_processes=400)
# make sure the index of gts and preds are the same
pred_dict = {i: p for i, p in zip(range(len(pred_list)), pred_list)} 

# get gt_list and prev_list
for idx in gts:
    gt = gts[idx]
    gt_list = [id2name[r] for r in gt['rec']]
    prev_list = [id2name[r] for r in gt['prev_entity']]
    pred_dict[idx]['gt_list'] = gt_list
    pred_dict[idx]['prev_list'] = prev_list

# save the intermediate results
os.makedirs(os.path.join(DIR, f'test_res/{alg}/{dataset}/{model}/intermediate/'), exist_ok=True)

extracted_path = os.path.join(DIR, f'test_res/{alg}/{dataset}/{model}/intermediate/extracted.jsonl')
with open(extracted_path, 'w') as f:
    # sorted keys
    for idx in sorted(pred_dict.keys()):
        json.dump(pred_dict[idx], f)
        f.write('\n')

test_res/sft_generated/inspired/llama3-2-1b-instruct/inspired_test_conv_entropy_ep5lr2e-5.jsonl


Processing:   0%|          | 0/228 [00:00<?, ?it/s]

not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)not enough values to unpack (expected 2, got 1)

not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2,

Processing: 100%|██████████| 228/228 [00:04<00:00, 49.37it/s]


In [54]:
! python evaluate.py --from_json {extracted_path}

253it [00:00, 373121.98it/s]
253it [00:00, 465216.53it/s]
253it [00:00, 402564.08it/s]
253it [00:00, 466648.60it/s]
Results are saved in /home/sagemaker-user/csbai/multiturn_rl/evaluation/zeroshot_test/test_res/sft_generated/inspired/llama3-2-1b-instruct/intermediate!


In [55]:
import pandas as pd

# Replace 'your_file.csv' with the actual path to your CSV file
df = pd.read_csv(f'test_res/{alg}/{dataset}/{model}/intermediate/filtered_True_exclude_seen_True/summary.csv')
print("Model: "+ model+" on Dataset: "+dataset)
print("recall@1_mean", df['recall@1_mean'][0])
print("recall@1_se", df['recall@1_se'][0])
print("recall@5_mean", df['recall@5_mean'][0])
print("recall@5_se", df['recall@5_se'][0])
# print("recall@10_mean", df['recall@10_mean'][0])
# print("recall@10_se", df['recall@10_se'][0])
# print("recall@20_mean", df['recall@20_mean'][0])
# print("recall@20_se", df['recall@20_se'][0])

Model: llama3-2-1b-instruct on Dataset: inspired
recall@1_mean 0.0331753554502369
recall@1_se 0.0123586775454078
recall@5_mean 0.0568720379146919
recall@5_se 0.0159817767082687
